In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import pearsonr
from adjustText import adjust_text
from matplotlib_venn import venn2

In [ ]:
de_and_go_results = pd.read_excel('data/DE_and_GO_results.xlsx', sheet_name=None, index_col=0)

# DE Correlation Overview

In [ ]:
results_df_given_comparison = {k: v for k, v in de_and_go_results.items() if k.startswith('P3') and 'T1_' in k}
keys = sorted(results_df_given_comparison.keys())
M = np.zeros((len(keys), len(keys)))
for i, k1 in enumerate(keys):
    print('.', end='')
    for j in range(i):
        k2 = keys[j]
        df1 = results_df_given_comparison[k1]
        df2 = results_df_given_comparison[k2]
        df = pd.merge(df1, df2, left_index=True, right_index=True)
        df.dropna(inplace=True)
        M[i, j] = abs(pearsonr(df['log2FoldChange_x'], df['log2FoldChange_y'])[0])
M = M + M.transpose()

In [ ]:
def sort_key(tup):
    idx, k = tup
    comp = k[len('P3_DE_T1_'):]
    if comp.endswith('P'):
        if 'EE' in k:
            return 0, 'B' if 'EEP' in k else 'A'
        return 0, k
    elif '_C-' in comp:
        return (1 if 'WT' in comp else 3 if '2PC' in comp else 2), k
    else:
        return (4 if '2PC' not in comp else 6 if '1PC' not in comp else 5), k
        
order = [idx for (idx, k) in sorted(enumerate(keys), key=sort_key)]
new_M = M[order, :]
new_M = new_M[:, order]

fig, ax = plt.subplots(figsize=(12, 9))
ms = ax.matshow(np.tril(new_M), cmap='Purples', vmin=0, vmax=1)
ax.grid(False)
cbar = plt.colorbar(ms)
cbar.set_label('Absolute Pearson Correlation')
ordered_keys = [keys[i] for i in order]
labels = [k[len('P3_DE_T1_'):] for k in ordered_keys]
ax.set_xticks(range(len(ordered_keys)))
ax.xaxis.set_ticks_position('bottom')
ax.set_xticklabels(labels, fontsize=8, rotation=90)
ax.set_yticks(range(len(ordered_keys)))
ax.set_yticklabels(labels, fontsize=8)
print()

# Supervolcano plots

In [ ]:
def supervolcano(
    df1,
    df2,
    label1='Test 1',
    label2='Test 2',
    top_n=6,
    padj_thresh=0.05,
    title='',
    figsize=(6, 6),
    annotate=True,
    save_path=None,
):
    """
    Plot signed -log10(padj) scatter using LFC sign for axis directionality.

    Parameters:
        df1, df2 (pd.DataFrame): pyDESeq2 results_df with 'padj' and 'log2FoldChange', indexed by gene.
        label1, label2 (str): Names of tests for axis labeling.
        top_n (int): Number of genes to annotate.
        padj_thresh (float): Significance threshold.
        title (str): Title of the plot.
        figsize (tuple): Size of the figure.
        annotate (bool): Whether to annotate top genes.
    """

    # Check columns
    for df, name in zip([df1, df2], [label1, label2]):
        if 'padj' not in df.columns or 'log2FoldChange' not in df.columns:
            raise ValueError(f"{name} must contain 'padj' and 'log2FoldChange'.")

    # Merge data
    merged = pd.merge(
        df1[['padj', 'log2FoldChange']],
        df2[['padj', 'log2FoldChange']],
        left_index=True,
        right_index=True,
        suffixes=(f'_{label1}', f'_{label2}')
    ).dropna()

    # Apply signed -log10(padj)
    def signed_log_pval(padj, lfc):
        return -np.log10(padj) * np.sign(lfc)

    merged[f'signed_log_padj_{label1}'] = signed_log_pval(
        merged[f'padj_{label1}'], merged[f'log2FoldChange_{label1}']
    )
    merged[f'signed_log_padj_{label2}'] = signed_log_pval(
        merged[f'padj_{label2}'], merged[f'log2FoldChange_{label2}']
    )

    # Filter significant
    sig = merged[
        (merged[f'padj_{label1}'] < padj_thresh) |
        (merged[f'padj_{label2}'] < padj_thresh)
    ].copy()

    # Classify LFC quadrant
    def classify(row):
        s1 = np.sign(row[f'log2FoldChange_{label1}'])
        s2 = np.sign(row[f'log2FoldChange_{label2}'])
        if s1 > 0 and s2 > 0:
            return 'Up in both'
        elif s1 < 0 and s2 < 0:
            return 'Down in both'
        elif s1 > 0 and s2 < 0:
            return f'Up in {label1} only'
        elif s1 < 0 and s2 > 0:
            return f'Up in {label2} only'
        else:
            return 'Mixed/neutral'

    sig['Quadrant'] = sig.apply(classify, axis=1)

    # Top genes to annotate
    if annotate:
        top_gene_indices = []
    
        for quadrant, group in sig.groupby('Quadrant'):
            group = group.copy()
    
            # Top n by padj in Test 1 (if significant)
            top_1 = group[group[f'padj_{label1}'] < padj_thresh].nsmallest(top_n, f'padj_{label1}')
            # Top n by padj in Test 2 (if significant)
            top_2 = group[group[f'padj_{label2}'] < padj_thresh].nsmallest(top_n, f'padj_{label2}')
    
            # Union of indices
            combined = top_1.index.union(top_2.index)
            top_gene_indices.extend(combined)
    
        # Final deduplicated annotation set
        top_genes = sig.loc[pd.Index(top_gene_indices).unique()]
    else:
        top_genes = pd.DataFrame()



    # Plot
    plt.figure(figsize=figsize)

    palette = {
        'Up in both': 'red',
        'Down in both': 'blue',
        f'Up in {label1} only': 'orange',
        f'Up in {label2} only': 'green',
        'Mixed/neutral': 'grey'
    }

    for cat, group in sig.groupby('Quadrant'):
        plt.scatter(
            group[f'signed_log_padj_{label1}'],
            group[f'signed_log_padj_{label2}'],
            label=cat,
            color=palette.get(cat, 'grey'),
            alpha=0.7,
            #edgecolor='k',
            s=40
        )

    # Axis labels
    plt.axhline(0, linestyle='--', color='black', lw=0.5)
    plt.axvline(0, linestyle='--', color='black', lw=0.5)
    plt.axhspan(np.log10(padj_thresh), -np.log10(padj_thresh), color='grey', alpha=0.2, zorder=-1)
    plt.axvspan(np.log10(padj_thresh), -np.log10(padj_thresh), color='grey', alpha=0.2, zorder=-1)
    axis_label = "$-\log_{10}p_{adj}\cdot sign(\mathrm{LFC})$"
    plt.xlabel(axis_label)
    plt.ylabel(axis_label)
    plt.title(f'X: {label1}, Y: {label2}')
    plt.grid(False)
    ax = plt.gca()
#    ax.set_xticklabels([xx.get_text().replace('−', '') for xx in ax.get_xticklabels()])
#    ax.set_yticklabels([yy.get_text().replace('−', '') for yy in ax.get_yticklabels()])
#    plt.legend(title='LFC Direction')

    # Annotate
    if annotate and not top_genes.empty:
        texts = []
        for gene, row in top_genes.iterrows():
            texts.append(
                plt.text(
                    row[f'signed_log_padj_{label1}'],
                    row[f'signed_log_padj_{label2}'],
                    gene,
                    fontsize=9,
                    weight='bold'
                )
            )
        adjust_text(texts, arrowprops=dict(arrowstyle='->', lw=0.5, color='black'))

    plt.tight_layout()

    if save_path:
        plt.savefig(save_path, dpi=300)

In [ ]:
k1 = 'P3_DE_T10_aEC_C-P'
k2 = 'P3_DE_T10_aEC_C-WT'
supervolcano(de_and_go_results[k1], de_and_go_results[k2], k1, k2)

In [ ]:
k1 = 'P3_DE_T10_aEC_C-P'
k2 = 'P3_DE_T10_aEC_WT-2PC'
supervolcano(de_and_go_results[k1], de_and_go_results[k2], k1, k2, top_n=3)

In [ ]:
k1 = 'P3_DE_ISC_Ant-Post_C-C'
k2 = 'P3_DE_ISC_Ant-Post_2PC-2PC'
supervolcano(de_and_go_results[k1], de_and_go_results[k2], k1, k2)

In [ ]:
k1 = 'LC_DE_ISC_Ant-Post_C-C'
k2 = 'LC_DE_ISC_Ant-Post_2PC-2PC'
supervolcano(de_and_go_results[k1], de_and_go_results[k2], k1, k2)

In [ ]:
gois = ['LamC', 'Prosalpha3']
alpha = 0.05
up_genes = {}
down_genes = {}
for goi in gois:
    goi_abv = goi[0] + goi[-1]
    k1 = f'{goi_abv}_DE_ISC_Ant-Post_2PC-2PC'
    k2 = f'{goi_abv}_DE_ISC_Ant-Post_C-C'
    a = de_and_go_results[k1].copy()
    b = de_and_go_results[k2].copy()
    joined = a.join(b[['padj']].rename(columns={'padj': f"{'padj'}_df2"}), how="left")
    up_in_df1 = (joined['log2FoldChange'] > 0) & (joined['padj'] < alpha)
    down_in_df1 = (joined['log2FoldChange'] < 0) & (joined['padj'] < alpha)
    ns_in_df2 = joined[f"{'padj'}_df2"].ge(alpha) | joined[f"{'padj'}_df2"].isna()
    mask = up_in_df1 & ns_in_df2
    up_genes[goi] = set(joined.loc[mask].index.tolist())
    mask = down_in_df1 & ns_in_df2
    down_genes[goi] = set(joined.loc[mask].index.tolist())
fig, ax = plt.subplots(figsize=(3, 2.5))
venn2([up_genes[goi] for goi in gois], set_labels=gois)
title = f'Higher in Posterior ISC'
ax.set_title(title)
fig, ax = plt.subplots(figsize=(3, 2.5))
venn2([down_genes[goi] for goi in gois], set_labels=gois)
title = f'Higher in Anterior ISC'
ax.set_title(title)
print()

# Selected Results

In [ ]:
celltypes = ['ISC-EB', 'aEC', 'pEC']

results_df_given_ct = {ct: de_and_go_results[f'P3_DE_T10_{ct}_WT-2PC'] for ct in celltypes}
genes_of_interest = [
    'dpn', 'Rpt3R', 'GstD2', 'GstE7', 
    'MtnA', 'MtnC', 'MtnD', 'ERp60', 'Hsc70-3', 'Calr', 
    'NAAT1', 'Npc2d', 'NLaz',
]
X = np.zeros((len(genes_of_interest), len(celltypes)))
for i, goi in enumerate(genes_of_interest):
    for j, ct in enumerate(celltypes):
        try:
            X[i, j] = results_df_given_ct[ct].loc[goi, 'log2FoldChange']
        except KeyError:
            pass

fig, ax = plt.subplots(figsize=(3, 3))
extreme = max(abs(X.min()), X.max())
m = ax.matshow(X, vmin=-extreme, vmax=extreme, cmap='bwr')
ax.set_xticks(range(len(celltypes)))
ax.set_xticklabels(celltypes, rotation=90)
ax.set_yticks(range(len(genes_of_interest)))
ax.set_yticklabels(genes_of_interest)
ax.xaxis.set_ticks_position('bottom')
ax.set_title(f'Prosalpha3 WT-2PC')
cbar = plt.colorbar(m)
cbar.set_label('LFC')

In [ ]:
go_terms_given_ct = {
    'ISC-EB': ['0004364', '0006749', '0000502', '0006511'],
    'aEC': ['0055065', '0046872', '0010038', '0034976', '0005783', '0006508'],
    'pEC': ['0015918', '0032367', '0015804', '0005416'],
}

n_prev = 0
all_idxs, all_descs = [], []
fig, ax = plt.subplots(figsize=(1.7, 4))
for ct in ['ISC-EB', 'aEC', 'pEC']:
    df = de_and_go_results[f'P3_GO_{ct}_WT-2PC']
    go_terms = [f'GO:{gt}' for gt in go_terms_given_ct[ct]]
    q_given_go = {} #go: df.loc[go, 'GO Name'] for go in go_terms}
    desc_given_go = {} #go: df.loc[go, 'qval'] for go in go_terms}
    for go in go_terms:
        if go in df.index:
            q_given_go[go] = df.loc[go, 'qval']
            desc_given_go[go] = df.loc[go, 'GO Name']
        else:
            print(go)
    go_terms = [gt for gt, q in sorted(q_given_go.items(), key=lambda tup: tup[1])]
    log_qs = [-np.log10(q_given_go[go]) for go in go_terms]
    idxs = -n_prev-np.arange(len(log_qs))
    ax.stem(idxs, log_qs, orientation='horizontal', basefmt='C0-')
    n_prev += len(log_qs) + 0.5
    all_idxs.extend(list(idxs))
    all_descs.extend([desc_given_go[gt] for gt in go_terms])
ax.set_yticks(all_idxs)
ax.set_yticklabels(all_descs)
ax.set_xlabel('$-log_{10} p_{adj}$')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)